In [1]:
!pip install sentence-transformers scikit-learn pandas numpy matplotlib

In [2]:
import pandas as pd
import numpy as np
import math
import time

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

In [3]:
chunks_df = pd.read_csv("chunks_final.csv")

print(chunks_df.head())

print("\nDataset Shape:")
print(chunks_df.shape)

   chunk_id   paper_id                                              title  \
0         1  paper_001  retrieval augmented generation for knowledge i...   
1         2  paper_001  retrieval augmented generation for knowledge i...   
2         3  paper_001  retrieval augmented generation for knowledge i...   
3         4  paper_001  retrieval augmented generation for knowledge i...   
4         5  paper_001  retrieval augmented generation for knowledge i...   

                                            pdf_file  page  chunk_number  \
0  retrieval_augmented_generation_for_knowledge_i...     1             1   
1  retrieval_augmented_generation_for_knowledge_i...     2             1   
2  retrieval_augmented_generation_for_knowledge_i...     3             1   
3  retrieval_augmented_generation_for_knowledge_i...     4             1   
4  retrieval_augmented_generation_for_knowledge_i...     5             1   

                                          chunk_text  
0  Retrieval-Augmented Ge

In [4]:
chunks_df = chunks_df.dropna(subset=['chunk_text'])

chunks_df = chunks_df.reset_index(drop=True)

texts = chunks_df['chunk_text'].astype(str).tolist()

print("Total chunks:", len(texts))

Total chunks: 1958


In [5]:
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

X_tfidf = vectorizer.fit_transform(texts)

print("TF-IDF Matrix Shape:")
print(X_tfidf.shape)

TF-IDF Matrix Shape:
(1958, 10000)


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings Shape:")
print(embeddings.shape)

Batches:   0%|          | 0/62 [00:00<?, ?it/s]

Embeddings Shape:
(1958, 384)


In [8]:
np.save("embeddings.npy", embeddings)

print("Embeddings saved")

Embeddings saved


In [9]:
def tfidf_search(query, top_k=5):

    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        X_tfidf
    ).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()

    results['score'] = similarities[top_indices]

    return results[
        [
            'chunk_id',
            'title',
            'page',
            'chunk_text',
            'score'
        ]
    ]

In [10]:
def dense_search(query, top_k=5):

    query_embedding = model.encode(
        query,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = np.dot(
        embeddings,
        query_embedding
    )

    top_indices = similarities.argsort()[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()

    results['score'] = similarities[top_indices]

    return results[
        [
            'chunk_id',
            'title',
            'page',
            'chunk_text',
            'score'
        ]
    ]

In [11]:
def hybrid_search(query, top_k=5, alpha=0.7):

    # TF-IDF scores
    query_vector = vectorizer.transform([query])
    tfidf_scores = cosine_similarity(query_vector, X_tfidf).flatten()

    # Dense scores
    query_embedding = model.encode(
        query, convert_to_numpy=True, normalize_embeddings=True
    )
    dense_scores = np.dot(embeddings, query_embedding)

    # Normalize both to [0, 1] before combining
    def minmax(arr):
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)

    tfidf_norm = minmax(tfidf_scores)
    dense_norm = minmax(dense_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * tfidf_norm

    top_indices = final_scores.argsort()[::-1][:top_k]

    results = chunks_df.iloc[top_indices].copy()
    results['tfidf_score'] = tfidf_scores[top_indices]
    results['dense_score'] = dense_scores[top_indices]
    results['final_score'] = final_scores[top_indices]

    return results[['chunk_id', 'title', 'page', 'chunk_text',
                     'tfidf_score', 'dense_score', 'final_score']]

In [12]:
query = "What is retrieval augmented generation?"

results = hybrid_search(
    query,
    top_k=5
)

results

,chunk_id,title,page,chunk_text,tfidf_score,dense_score,final_score
1760,1865,retrieval augmented generation evaluation in t...,16,"16 Front. Comput. Sci., 2025, 0(0): 1–18 82. C...",0.404122,0.550008,0.872290
1761,1866,retrieval augmented generation evaluation in t...,16,Chen E. Crud-rag: A comprehensive chinese benc...,0.329200,0.607619,0.867766
711,752,hybrid retrieval for hallucination mitigation ...,14,January 2025 org/CorpusID:266359151. [22] Li S...,0.282363,0.631964,0.854587
41,42,retrieval augmented generation for large langu...,17,"preprint arXiv:2212.14024, 2022. [24] Z. Jiang...",0.198160,0.694007,0.847104
1897,2007,can llms evaluate retrieval augmented generation,8,"[31] J. Liu, R. Ding, L. Zhang, P. Xie, and F....",0.253480,0.637787,0.838310


In [13]:
for idx, row in results.iterrows():

    print("=" * 100)

    print("TITLE:", row['title'])

    print("PAGE:", row['page'])

    print("FINAL SCORE:", row['final_score'])

    print()

    print(row['chunk_text'][:800])

    print()

TITLE: retrieval augmented generation evaluation in the era of large language
PAGE: 16
FINAL SCORE: 0.8722898237434287

16 Front. Comput. Sci., 2025, 0(0): 1–18 82. Cheng X, Wang X, Zhang X, Ge T, Chen S Q, Wei F, Zhang H, Zhao D. xrag: Extreme context compression for retrieval-augmented gen- eration with one token. arXiv preprint arXiv:2405.13792, 2024 83. Asai A, Wu Z, Wang Y, Sil A, Hajishirzi H. Self-rag: Learning to re- trieve, generate, and critique through self-reflection. In: The Twelfth International Conference on Learning Representations. 2023 84. Trivedi H, Balasubramanian N, Khot T, Sabharwal A. Interleav- ing retrieval with chain-of-thought reasoning for knowledge-intensive multi-step questions. In: ACL (1). 2023, 10014–10037 85. Chen J, Lin H, Han X, Sun L. Benchmarking large language mod- els in retrieval-augmented generation. In: Proceedings of the AAAI Conference on Artificial Intelligence.

TITLE: retrieval augmented generation evaluation in the era of large language


In [14]:
def recall_at_k(
    retrieved_ids,
    relevant_id,
    k=5
):

    return int(
        relevant_id in retrieved_ids[:k]
    )


def ndcg_at_k(
    retrieved_ids,
    relevant_id,
    k=5
):

    dcg = 0

    for i, doc_id in enumerate(retrieved_ids[:k]):

        if doc_id == relevant_id:

            dcg = 1 / math.log2(i + 2)

            break

    idcg = 1

    return dcg / idcg

In [15]:
# Generate proper gold queries by using the actual chunk content as the query
# Pick chunks that have meaningful descriptive text (not just references)

# Filter out reference/bibliography chunks
mask = (
    ~chunks_df['chunk_text'].str.contains(r'\[\d+\]', regex=True) &  # no citations like [1]
    ~chunks_df['chunk_text'].str.contains(r'arXiv', case=False) &     # no arxiv refs
    (chunks_df['chunk_text'].str.len() > 200)                         # meaningful length
)

clean_chunks = chunks_df[mask].reset_index(drop=True)
print(f"Clean chunks available: {len(clean_chunks)}")

# Sample 20
sample_chunks = clean_chunks.sample(20, random_state=42).reset_index(drop=True)

# Use first 100 chars of each chunk as the query (close to the actual content)
gold_queries = pd.DataFrame({
    "query": sample_chunks['chunk_text'].str.slice(0, 100).tolist(),
    "relevant_chunk_id": sample_chunks['chunk_id'].astype(str).tolist()
})

gold_queries.to_csv("gold_queries.csv", index=False)
print("\nNew gold queries:")
print(gold_queries[['query', 'relevant_chunk_id']].to_string())

Clean chunks available: 974

New gold queries:
                                                                                                   query relevant_chunk_id
0   Technical Report Table 8: Statistical information of the datasets utilized in this paper. Dataset Na               355
1   Published as a conference paper at ICLR 2026 Figure 3: The overall framework of GraphRAG-Bench. It c              1125
2   and metadata- based approaches, which can be defined as follows: αi = a·scorekey(q, Fj)+(1−α)·maxθj∈               309
3   A Appendix A.1 Instructions A.1.1 Web Search Please write a passage to answer the question Question:               933
4   Improving language models by retrieving from trillions of tokens Algorithm 1: Overview of Retro mode               144
5   function S(.) rather than exploiting the last-token representation for relevance scoring. score(.) i               873
6   January 2025 Figure 3. Overall Performance in Mitigating Hallucinations on HaluBenchsmal

In [16]:
queries_df = pd.read_csv("gold_queries.csv")

recalls = []
ndcgs = []
latencies = []

for _, row in queries_df.iterrows():

    query = row['query']
    relevant_id = str(row['relevant_chunk_id'])

    start = time.time()

    results = hybrid_search(
        query,
        top_k=5
    )

    end = time.time()

    latencies.append(end - start)

    retrieved_ids = results['chunk_id'].astype(str).tolist()

    recalls.append(
        recall_at_k(
            retrieved_ids,
            relevant_id
        )
    )

    ndcgs.append(
        ndcg_at_k(
            retrieved_ids,
            relevant_id
        )
    )

print("Recall@5:", np.mean(recalls))
print("NDCG@5:", np.mean(ndcgs))
print("P95 Latency:", np.percentile(latencies, 95))

Recall@5: 0.9
NDCG@5: 0.7955285910760186
P95 Latency: 0.01736855506896973


In [17]:
metrics_df = pd.DataFrame({

    'Method': ['Hybrid Retrieval'],

    'Recall@5': [np.mean(recalls)],

    'NDCG@5': [np.mean(ndcgs)],

    'P95 Latency': [
        np.percentile(latencies, 95)
    ]
})

metrics_df.to_csv(
    "evaluation_metrics.csv",
    index=False
)

metrics_df

,Method,Recall@5,NDCG@5,P95 Latency
0,Hybrid Retrieval,0.9,0.795529,0.017369


In [18]:
# ============================================================
# CREATE retrieval_results.csv FOR PERSON 4: RIVER + ADWIN
# ============================================================

import pandas as pd
import numpy as np
import time

queries_df = pd.read_csv("gold_queries.csv")

# Add query_id if it does not exist
if "query_id" not in queries_df.columns:
    queries_df.insert(0, "query_id", [f"Q{i+1}" for i in range(len(queries_df))])

all_rows = []

TOP_K = 5
ALPHA = 0.7  # replace this with Person 3's best alpha if available

for _, qrow in queries_df.iterrows():

    query_id = qrow["query_id"]
    query = qrow["query"]
    relevant_chunk_id = str(qrow["relevant_chunk_id"])

    start_time = time.time()

    results = hybrid_search(
        query=query,
        top_k=TOP_K,
        alpha=ALPHA
    )

    latency = time.time() - start_time

    retrieved_ids = results["chunk_id"].astype(str).tolist()

    helpful = int(relevant_chunk_id in retrieved_ids)

    for rank, (_, rrow) in enumerate(results.iterrows(), start=1):

        chunk_id = str(rrow["chunk_id"])

        is_relevant = int(chunk_id == relevant_chunk_id)

        all_rows.append({
            "query_id": query_id,
            "query": query,
            "rank": rank,
            "chunk_id": chunk_id,
            "relevant_chunk_id": relevant_chunk_id,
            "tfidf_score": float(rrow["tfidf_score"]),
            "dense_score": float(rrow["dense_score"]),
            "hybrid_score": float(rrow["final_score"]),
            "is_relevant": is_relevant,
            "helpful": helpful,
            "latency": latency,
            "alpha": ALPHA
        })

retrieval_results_df = pd.DataFrame(all_rows)

retrieval_results_df.to_csv("retrieval_results.csv", index=False)

print("retrieval_results.csv saved")
print(retrieval_results_df.head(10))
print("Shape:", retrieval_results_df.shape)

retrieval_results.csv saved
  query_id                                              query  rank chunk_id  \
0       Q1  Technical Report Table 8: Statistical informat...     1      355   
1       Q1  Technical Report Table 8: Statistical informat...     2     1392   
2       Q1  Technical Report Table 8: Statistical informat...     3      338   
3       Q1  Technical Report Table 8: Statistical informat...     4     1997   
4       Q1  Technical Report Table 8: Statistical informat...     5      699   
5       Q2  Published as a conference paper at ICLR 2026 F...     1     1137   
6       Q2  Published as a conference paper at ICLR 2026 F...     2     1135   
7       Q2  Published as a conference paper at ICLR 2026 F...     3     1016   
8       Q2  Published as a conference paper at ICLR 2026 F...     4     1129   
9       Q2  Published as a conference paper at ICLR 2026 F...     5     1069   

  relevant_chunk_id  tfidf_score  dense_score  hybrid_score  is_relevant  \
0              